In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

In [ ]:
from pathlib import Path
import pandas as pd

# This tells Python to go up one folder from 'notebooks' to 'Retail-Analytics'
BASE_DIR = Path("../") 

CLEANED = BASE_DIR / "data" / "cleaned"
PROCESSED = BASE_DIR / "data" / "processed"

# Now try loading them again
customers = pd.read_csv(CLEANED / "customers_cleaned.csv")
orders = pd.read_csv(CLEANED / "orders_cleaned.csv")
order_items = pd.read_csv(CLEANED / "order_items_cleaned.csv")

In [ ]:
from pathlib import Path
import pandas as pd

# Hardcode the exact path to your main project folder
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Define the paths
CLEANED = BASE_DIR / "data" / "cleaned"
PROCESSED = BASE_DIR / "data" / "processed"

customers = pd.read_csv(CLEANED / "customers_cleaned.csv")
orders = pd.read_csv(CLEANED / "orders_cleaned.csv")
order_items = pd.read_csv(CLEANED / "order_items_cleaned.csv")
products = pd.read_csv(CLEANED / "products_cleaned.csv")
payments = pd.read_csv(CLEANED / "payments_cleaned.csv")
reviews = pd.read_csv(CLEANED / "reviews_cleaned.csv")
sellers = pd.read_csv(CLEANED / "sellers_cleaned.csv")
category = pd.read_csv(CLEANED/"category_translation_cleaned.csv")

In [ ]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

In [ ]:
customer_orders = (
    orders.groupby("customer_id")
    .size()
    .rename("customer_total_orders")
)

customers = customers.merge(
    customer_orders,
    on="customer_id",
    how="left"
)

customers["customer_total_orders"] = customers["customer_total_orders"].fillna(0)

In [ ]:
# 1. Group the orders and convert directly into a DataFrame
customer_orders = (
    orders.groupby("customer_id")
    .size()
    .reset_index(name="customer_total_orders") # <-- This is the magic fix!
)

# 2. Merge the two DataFrames
customers = customers.merge(
    customer_orders,
    on="customer_id",
    how="left"
)

# 3. Fill the missing values
customers["customer_total_orders"] = customers["customer_total_orders"].fillna(0)

In [ ]:
import numpy as np

# 1. Merge orders and order_items so we know which customer bought which items
customer_purchases = orders.merge(order_items, on="order_id", how="left")

# 2. Group by customer_id and sum the prices of all their items
customer_spent = (
    customer_purchases.groupby("customer_id")["price"]
    .sum()
    .reset_index(name="customer_total_spent")
)

# 3. Merge this new total spent column into your main customers dataframe
customers = customers.merge(
    customer_spent,
    on="customer_id",
    how="left"
)

# 4. Fill missing values with 0 (for customers who didn't buy anything)
customers["customer_total_spent"] = customers["customer_total_spent"].fillna(0)

# 5. NOW you can safely calculate the average order value!
customers["customer_avg_order_value"] = np.where(
    customers["customer_total_orders"] > 0,                                   
    customers["customer_total_spent"] / customers["customer_total_orders"],   
    0                                                                         
)

In [ ]:
customers["repeat_customer"] = (
    customers["customer_total_orders"] > 1
).astype(int)

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_purchase_timestamp"]
).dt.days

In [ ]:
orders["approval_days"] = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.days

In [ ]:
orders["carrier_days"] = (
    orders["order_delivered_carrier_date"] -
    orders["order_approved_at"]
).dt.days

In [ ]:
orders["carrier_days"] = (
    orders["order_delivered_carrier_date"] -
    orders["order_approved_at"]
).dt.days

In [ ]:
orders["delivery_delay"] = (
    orders["order_delivered_customer_date"] -
    orders["order_estimated_delivery_date"]
).dt.days

In [ ]:
orders["late_delivery"] = (
    orders["delivery_delay"] > 0
).astype(int)

In [ ]:
orders["year"] = orders["order_purchase_timestamp"].dt.year

orders["month"] = orders["order_purchase_timestamp"].dt.month

orders["quarter"] = orders["order_purchase_timestamp"].dt.quarter

orders["weekday"] = orders["order_purchase_timestamp"].dt.day_name()

orders["hour"] = orders["order_purchase_timestamp"].dt.hour

orders["weekend"] = (
    orders["weekday"].isin(["Saturday","Sunday"])
).astype(int)

In [ ]:
products["product_volume_cm3"] = (

products["product_length_cm"] *

products["product_width_cm"] *

products["product_height_cm"]

)

In [ ]:
products["density"] = (

products["product_weight_g"] /

products["product_volume_cm3"]

)

In [ ]:
products.replace([np.inf,-np.inf],np.nan,inplace=True)

In [ ]:
products["heavy_product"] = (
products["product_weight_g"]>5000
).astype(int)

In [ ]:
products["large_product"] = (
products["product_volume_cm3"]>
products["product_volume_cm3"].median()
).astype(int)

In [ ]:
order_items["revenue"] = order_items["price"]

In [ ]:
order_items["freight_ratio"] = (

order_items["freight_value"]

/

order_items["price"]

)*100

In [ ]:
order_value = (

order_items.groupby("order_id")["price"]

.sum()

.rename("order_value")

)

order_items = order_items.merge(

order_value,

on="order_id"

)

In [ ]:
basket = (

order_items.groupby("order_id")

.size()

.rename("basket_size")

)

order_items = order_items.merge(

basket,

on="order_id"

)

In [ ]:
seller_rev = (

order_items.groupby("seller_id")["price"]

.sum()

.rename("seller_revenue")

)

order_items = order_items.merge(

seller_rev,

on="seller_id"

)

In [ ]:
product_rev = (

order_items.groupby("product_id")["price"]

.sum()

.rename("product_revenue")

)

order_items = order_items.merge(

product_rev,

on="product_id"

)

In [ ]:
payments["credit_card"] = (
payments["payment_type"]=="credit_card"
).astype(int)

In [ ]:
payments["voucher"] = (
payments["payment_type"]=="voucher"
).astype(int)

In [ ]:
payments["boleto"] = (
payments["payment_type"]=="boleto"
).astype(int)

In [ ]:
payments["debit_card"] = (
payments["payment_type"]=="debit_card"
).astype(int)

In [ ]:
payments["installment_group"] = pd.cut(

payments["payment_installments"],

bins=[0,1,3,6,12,30],

labels=[

"Single",

"2-3",

"4-6",

"7-12",

"12+"

]

)

In [ ]:
reviews["positive_review"] = (
reviews["review_score"]>=4
).astype(int)

In [ ]:
reviews["negative_review"] = (
reviews["review_score"]<=2
).astype(int)

In [ ]:
reviews["review_length"] = (

reviews["review_comment_message"]

.fillna("")

.str.len()

)

In [ ]:
seller_orders = (
order_items.groupby("seller_id")
.size()
.rename("seller_total_orders")
)

sellers = sellers.merge(
seller_orders,
on="seller_id",
how="left"
)

sellers["seller_total_orders"] = sellers["seller_total_orders"].fillna(0)

In [ ]:
customers.to_csv(PROCESSED/"customers_features.csv",index=False)

orders.to_csv(PROCESSED/"orders_features.csv",index=False)

products.to_csv(PROCESSED/"products_features.csv",index=False)

order_items.to_csv(PROCESSED/"order_items_features.csv",index=False)

payments.to_csv(PROCESSED/"payments_features.csv",index=False)

reviews.to_csv(PROCESSED/"reviews_features.csv",index=False)

sellers.to_csv(PROCESSED/"sellers_features.csv",index=False)